# 📈 BrandMind AI — Week 7
## Smart Social & Brand Campaign Studio
**Tools:** Pandas · scikit-learn · Plotly

In [ ]:
!pip install pandas scikit-learn plotly matplotlib numpy joblib -q

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib, os, zipfile
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('Ready')

## 1. Data Preparation & Analysis

In [ ]:
np.random.seed(42)
n = 2000
platforms = ['Instagram','LinkedIn','Twitter','Facebook','TikTok']
regions   = ['North America','Europe','Asia','India','Global']
camp_types= ['Image','Video','Carousel','Story','Reel']
goals     = ['awareness','engagement','conversion','lead_gen']

df = pd.DataFrame({
    'platform':      np.random.choice(platforms, n),
    'region':        np.random.choice(regions, n),
    'campaign_type': np.random.choice(camp_types, n),
    'goal':          np.random.choice(goals, n),
    'budget':        np.random.randint(500, 50000, n),
    'duration_days': np.random.randint(3, 30, n),
    'audience_size': np.random.randint(1000, 500000, n),
})

# Realistic targets with noise
df['ctr']        = (np.random.uniform(0.5, 8.0, n) + (df['platform']=='TikTok').astype(float)*1.5 + (df['campaign_type']=='Video').astype(float)*0.8).clip(0.1, 12)
df['roi']        = (np.random.uniform(0.8, 5.5, n) + np.log1p(df['budget'])*0.1).clip(0.3, 8)
df['engagement'] = (np.random.uniform(2, 14, n) + (df['platform']=='Instagram').astype(float)*1.2).clip(0.5, 20)

print(df.shape)
print(df.describe().round(2))

## 2. Predictive Modelling — CTR, ROI, Engagement

In [ ]:
cat_cols = ['platform','region','campaign_type','goal']
num_cols = ['budget','duration_days','audience_size']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])

X = df[cat_cols + num_cols]
results = {}

for target in ['ctr', 'roi', 'engagement']:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Random Forest
    rf_pipe = Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(n_estimators=200, random_state=42))])
    rf_pipe.fit(X_train, y_train)
    rf_preds = rf_pipe.predict(X_test)
    rf_rmse  = np.sqrt(mean_squared_error(y_test, rf_preds))
    rf_r2    = r2_score(y_test, rf_preds)

    results[target] = {'rf_rmse': rf_rmse, 'rf_r2': rf_r2, 'model': rf_pipe}
    joblib.dump(rf_pipe, f'models/campaign_{target}_rf.pkl')
    print(f'{target.upper():12s} | RMSE: {rf_rmse:.4f} | R2: {rf_r2:.4f}')

print('\nAll models saved to models/')

## 3. Campaign Generation + Regional Insights

In [ ]:
# Regional engagement analysis
region_stats = df.groupby('region')[['ctr','roi','engagement']].mean().round(3)
print('Regional Performance Averages:')
print(region_stats)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_labels = ['CTR (%)', 'ROI (x)', 'Engagement (%)']
colors = ['#1a1a18','#b8975a','#2d4a3e']

for ax, col, label, color in zip(axes, ['ctr','roi','engagement'], metrics_labels, colors):
    region_stats[col].plot(kind='barh', ax=ax, color=color, alpha=0.85)
    ax.set_title(f'Avg {label} by Region', fontweight='bold')
    ax.set_xlabel(label)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Regional Campaign Performance Insights', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/regional_engagement.png', dpi=150); plt.show()
print('Saved: outputs/regional_engagement.png')

## 4. Downloadable Campaign Kit ZIP

In [ ]:
def build_campaign_kit(company, platform, slogan, caption, metrics, palette_colors):
    os.makedirs('outputs/campaign_kit', exist_ok=True)
    # Campaign copy
    with open('outputs/campaign_kit/campaign_copy.txt', 'w') as f:
        f.write(f'BRAND CAMPAIGN KIT — {company.upper()}\n')
        f.write('='*50 + '\n\n')
        f.write(f'Platform: {platform}\n')
        f.write(f'Slogan: "{slogan}"\n\n')
        f.write(f'Caption:\n{caption}\n\n')
        f.write(f'Predicted Metrics:\n')
        for k, v in metrics.items():
            f.write(f'  {k}: {v}\n')

    # Colour palette
    with open('outputs/campaign_kit/palette.csv', 'w') as f:
        f.write('role,hex\n')
        roles = ['Primary','Secondary','Background','Neutral']
        for role, hex_c in zip(roles, palette_colors):
            f.write(f'{role},{hex_c}\n')

    # ZIP everything
    with zipfile.ZipFile('outputs/campaign_kit.zip', 'w') as zf:
        for fn in os.listdir('outputs/campaign_kit'):
            zf.write(f'outputs/campaign_kit/{fn}', fn)
    print('Campaign kit ZIP created: outputs/campaign_kit.zip')

build_campaign_kit(
    company='BrandMind',
    platform='Instagram',
    slogan='Intelligence meets identity.',
    caption='Redefining what it means to build a brand. Powered by AI. ✨\n\n#BrandMind #AI #Branding',
    metrics={'Predicted CTR': '4.2%', 'Estimated ROI': '3.1x', 'Engagement Rate': '7.8%'},
    palette_colors=['#1a1a18','#b8975a','#f5f0e8','#6b6a65']
)

## Week 7 Complete ✅
- ✅ Marketing data loaded, cleaned, encoded
- ✅ Random Forest CTR/ROI/engagement prediction with RMSE
- ✅ Regional engagement insights chart
- ✅ Campaign kit ZIP download
- ✅ Models saved (.pkl)